# rfahrur6045-testing.ipynb
## Testing prediction request ke Hugging Face Space

Notebook ini digunakan untuk menguji endpoint deployment machine learning yang sudah berjalan di Hugging Face Spaces.

Notebook ini dijalankan dari lokal untuk mengirim prediction request ke cloud, lalu membaca respons prediksi dari model yang sudah dideploy.

## 1. Setup

Cell ini menyiapkan library, URL Space, dan urutan fitur yang harus dikirim ke endpoint.

In [1]:
%pip install gradio-client>=0.2.9

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kfp 2.0.1 requires google-auth<3,>=1.6.1, which is not installed.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os

import pandas as pd
from gradio_client import Client

SPACE_PAGE_URL = "https://huggingface.co/spaces/rfahrur6045/mlops-final"
SPACE_ENDPOINT = os.getenv("SPACE_ENDPOINT", "https://rfahrur6045-mlops-final.hf.space")
HF_TOKEN = os.getenv("HF_TOKEN")

FEATURE_ORDER = [
    "Tenure",
    "CityTier",
    "WarehouseToHome",
    "HourSpendOnApp",
    "NumberOfDeviceRegistered",
    "SatisfactionScore",
    "NumberOfAddress",
    "Complain",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
    "CashbackAmount",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "Gender",
    "PreferedOrderCat",
    "MaritalStatus",
]

print("Space page URL:", SPACE_PAGE_URL)
print("Space endpoint :", SPACE_ENDPOINT)
print("Feature count  :", len(FEATURE_ORDER))

Space page URL: https://huggingface.co/spaces/rfahrur6045/mlops-final
Space endpoint : https://rfahrur6045-mlops-final.hf.space
Feature count  : 18


## 2. Sample Input

Cell ini menyiapkan dua skenario pelanggan untuk diuji: high risk dan low risk.

In [ ]:
high_risk = {
    "Tenure": 2,
    "CityTier": 1,
    "WarehouseToHome": 45.0,
    "HourSpendOnApp": 1.0,
    "NumberOfDeviceRegistered": 6,
    "SatisfactionScore": 1,
    "NumberOfAddress": 9,
    "Complain": 1,
    "OrderAmountHikeFromlastYear": 11.0,
    "CouponUsed": 1.0,
    "OrderCount": 2.0,
    "DaySinceLastOrder": 15.0,
    "CashbackAmount": 120.9,
    "PreferredLoginDevice": "Mobile Phone",
    "PreferredPaymentMode": "Cash on Delivery",
    "Gender": "Male",
    "PreferedOrderCat": "Mobile Phone",
    "MaritalStatus": "Single",
}

low_risk = {
    "Tenure": 24,
    "CityTier": 1,
    "WarehouseToHome": 10.0,
    "HourSpendOnApp": 5.0,
    "NumberOfDeviceRegistered": 2,
    "SatisfactionScore": 5,
    "NumberOfAddress": 2,
    "Complain": 0,
    "OrderAmountHikeFromlastYear": 25.0,
    "CouponUsed": 8.0,
    "OrderCount": 15.0,
    "DaySinceLastOrder": 2.0,
    "CashbackAmount": 280.0,
    "PreferredLoginDevice": "Computer",
    "PreferredPaymentMode": "Credit Card",
    "Gender": "Female",
    "PreferedOrderCat": "Laptop & Accessory",
    "MaritalStatus": "Married",
}

print("Skenario high risk dan low risk siap diuji.")

Skenario high risk dan low risk siap diuji.


## 3. Fungsi Prediction Request

Dua URL yang muncul di Network (`/gradio_api/queue/join` dan `/gradio_api/queue/data`) adalah jalur queue internal Gradio. Untuk pengujian dari notebook, gunakan `gradio_client.Client` supaya client yang menangani queue tersebut.

In [4]:
def to_model_input(instance):
    return [instance[feature] for feature in FEATURE_ORDER]


def predict_churn(instance, endpoint=SPACE_ENDPOINT, token=HF_TOKEN):
    client = Client(endpoint, token=token)
    return client.predict(*to_model_input(instance), api_name="/predict_churn")


def extract_prediction(response_body):
    return response_body


print("Fungsi predict_churn siap digunakan.")

Fungsi predict_churn siap digunakan.


## 4. Uji Prediksi - High Risk

Cell ini mengirim satu request prediksi untuk pelanggan berisiko churn tinggi.

In [5]:
high_response = predict_churn(high_risk)
high_result = extract_prediction(high_response)
high_result

Loaded as API: https://rfahrur6045-mlops-final.hf.space/


{'probability': 0.9534,
 'prediction': 'Churn',
 'model_dir': '/app/serving_model/rfahrur6045-pipeline'}

## 5. Uji Prediksi - Low Risk

Cell ini mengirim satu request prediksi untuk pelanggan berisiko churn rendah.

In [6]:
low_response = predict_churn(low_risk)
low_result = extract_prediction(low_response)
low_result

Loaded as API: https://rfahrur6045-mlops-final.hf.space/


{'probability': 0.0008,
 'prediction': 'Not Churn',
 'model_dir': '/app/serving_model/rfahrur6045-pipeline'}

## 6. Ringkasan Hasil

Gunakan cell ini untuk menuliskan kesimpulan singkat setelah hasil respons dari Space muncul. Jika respons berisi probabilitas, ubah menjadi label churn / tidak churn sesuai kebutuhan laporan submission.

In [7]:
summary = pd.DataFrame([
    {"scenario": "high_risk", "response": str(high_result)},
    {"scenario": "low_risk", "response": str(low_result)},
])
summary

,scenario,response
0,high_risk,"{'probability': 0.9534, 'prediction': 'Churn',..."
1,low_risk,"{'probability': 0.0008, 'prediction': 'Not Chu..."
